# 04 · Loaders, splitters, vector stores and retrievers

After this notebook you can load a folder of documents into LangChain `Document`s, chunk them (by characters and by
Markdown headings), embed them, index them in `InMemoryVectorStore` and FAISS (and save/load the index safely), turn a
store into a retriever (similarity, MMR, score threshold, metadata filters), add BM25 and a hybrid `EnsembleRetriever`,
and write your own `Embeddings` class.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · LCEL](02_lcel_runnables_and_output_parsers.ipynb) · theory without a framework: [vector search](../09_rag/03_vector_search_faiss_hnsw.ipynb), [hybrid search](../09_rag/04_hybrid_search_and_rrf.ipynb)

## Why this matters in interviews

- The indexing half of RAG is where most RAG systems actually break (bad chunks, missing metadata, wrong scores).
  Interviewers want you to name each component and its knobs.
- "Implement a custom `Embeddings` class" and "what does a retriever return?" test whether you know the interfaces,
  not just the tutorial.
- Hybrid search (BM25 + dense, fused with RRF), MMR and metadata filtering are the standard follow-ups to "semantic
  search is not enough".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg03` | What are chunks, and why not just index whole documents? |
| `rg41` | What is document-aware chunking and why is the heading path so valuable? |
| `rg44` | What goes in a chunk's metadata, and why is it not optional? |
| `rg10` | What is metadata filtering and why is it more important than it sounds? |
| `rg07` | What is hybrid search and how do you combine the two lanes? |
| `rg45` | What is reciprocal rank fusion and why is it the default? |
| `ve24` | What is MMR (maximal marginal relevance) and when do you want it? |
| `rg14` | Must the query and the documents use the same embedding model? What if they do not? |
| `rg49` | What is the difference between retrieval score and confidence? |
| `rg33` | What is a vector database and do you always need one? |

## Setup

The real embedding code is `OpenAIEmbeddings(model="text-embedding-3-small")` from `langchain_openai`;
`langchain_embeddings()` returns exactly that, wired to OpenAI, Ollama (`nomic-embed-text`) or the offline stand-in.
Offline embeddings are hashed bag-of-words with a small synonym table (vacation ≈ holiday, password ≈ credentials):
word overlap works, deep semantics does not — the queries below are chosen so that both behave sensibly.

**An import-time gotcha.** `langchain_text_splitters` (also imported by `langchain_classic.retrievers`) imports
`transformers` and `sentence_transformers` *if they are installed*, only to support token-based splitters. In this
venv that pulls in PyTorch and can add 10–60 s to the first import. We use neither, so the first cell tells Python
they are unavailable — remove those two lines if you want the Hugging Face splitters.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_embeddings, OFFLINE

for _heavy in ("sentence_transformers", "transformers"):      # skip optional heavy imports (see above)
    sys.modules.setdefault(_heavy, None)

import re, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
from langchain_core.documents import Document

## 1. `Document`: text plus metadata

Everything in LangChain's retrieval stack passes around `Document(page_content=str, metadata=dict, id=...)`.
Metadata is not decoration: it is what you filter on (tenant, permissions, product), what you cite (source, section)
and how you delete or update a file's chunks later.

We write a tiny engineering handbook — four Markdown files — into a temp folder (the same handbook as the
[LangChain course](../../langchain/index.html), chapter 7, plus a few sections).

In [ ]:
KB = {
"runbook.md": """# Runbook

## Deploys

All production deploys require two approvals and must land before 16:00 UTC on weekdays. Friday deploys are blocked unless the change is tagged hotfix. Every deploy is announced in the releases channel.

## Rollback

To roll back, run `deployctl rollback <service> --to <sha>`. Rollback does not revert database migrations; those need a forward fix. After a rollback, open an incident if customers were affected.
""",
"handbook.md": """# Handbook

## On-call

On-call rotations are one week, Monday 09:00 to Monday 09:00. Acknowledge a page within 5 minutes and escalate to the secondary after 15 minutes.

## Incidents

Any customer-visible outage over 5 minutes needs an incident channel, an incident lead and a written postmortem within 48 hours. The postmortem is blameless and lists follow-up actions with owners.

## Leave

Employees get 25 days of paid holiday per year. Holiday requests go through the HR portal at least two weeks ahead.
""",
"infra.md": """# Infrastructure

## Databases

The primary Postgres cluster runs version 16.2 with a connection pool ceiling of 80 per service. Raising the ceiling requires a capacity review.

## Kubernetes

Clusters are upgraded one minor version at a time, never skipping. Upgrades happen in staging first, with a 72-hour soak.
""",
"security.md": """# Security

## Access

Production database access is read-only by default. Write access is granted per incident and expires automatically after 4 hours.

## Secrets

Secrets and credentials live in the vault and are injected at runtime. A secret committed to git must be rotated even if the commit was reverted.

## Error codes

Error AUTH-4012 means the service token has expired. Request a new token from the vault with `vault token renew`.
""",
}
kb_dir = Path(tempfile.mkdtemp()) / "kb"
kb_dir.mkdir()
for name, text in KB.items():
    (kb_dir / name).write_text(text, encoding="utf-8")
(kb_dir.parent / "services.csv").write_text("service,owner,tier\npayments,team-a,1\nsearch,team-b,2\n", encoding="utf-8")
print(sorted(p.name for p in kb_dir.iterdir()))

## 2. Loaders

A loader turns a source (file, folder, URL, database, Notion...) into `Document`s with a `source` in the metadata.
`TextLoader` reads one file; `DirectoryLoader` globs a folder and applies a loader class to each file. Loaders also
have `lazy_load()`, a generator, for corpora that do not fit in memory. (They live in `langchain_community`, which now
warns that it is being sunset in favour of standalone integration packages — the classes still work in 1.x.)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)          # the langchain-community sunset notice
    from langchain_community.document_loaders import TextLoader, DirectoryLoader, CSVLoader

one = TextLoader(str(kb_dir / "runbook.md"), encoding="utf-8").load()
print("TextLoader     :", len(one), "doc |", one[0].metadata["source"].split("\\")[-1].split("/")[-1],
      "|", repr(one[0].page_content[:40]))

loader = DirectoryLoader(str(kb_dir), glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
docs = sorted(loader.load(), key=lambda d: d.metadata["source"])
print("DirectoryLoader:", len(docs), "docs:", [Path(d.metadata["source"]).name for d in docs])

rows = CSVLoader(str(kb_dir.parent / "services.csv"), encoding="utf-8").load()
print("CSVLoader      :", len(rows), "docs (one per row) ->", repr(rows[0].page_content), rows[0].metadata["row"])
assert len(docs) == 4 and len(rows) == 2

## 3. Splitters

**In plain English:** we cut documents into chunks small enough that (a) one chunk is about one thing, so its
embedding means something, and (b) a few chunks fit into the prompt. Too big: the embedding is a blur and you pay for
irrelevant text. Too small: the answer is split across chunks and loses its context.

`RecursiveCharacterTextSplitter` tries separators in order — paragraph `"\n\n"`, line `"\n"`, space, character — so it
cuts at the most natural boundary that keeps chunks under `chunk_size` characters, with `chunk_overlap` characters
repeated between neighbours so a sentence cut in half still appears whole somewhere.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

for size, overlap in [(400, 0), (150, 30), (80, 20)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap, add_start_index=True)
    pieces = splitter.split_documents(docs)
    lengths = [len(p.page_content) for p in pieces]
    print(f"chunk_size={size:3d} overlap={overlap:2d} -> {len(pieces):2d} chunks, max {max(lengths)} chars")
    assert max(lengths) <= size

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30, add_start_index=True)
char_chunks = splitter.split_documents(docs)
print("\nseparators tried in order:", splitter._separators)
print("a chunk's metadata      :", {k: (Path(v).name if k == "source" else v) for k, v in char_chunks[3].metadata.items()})
print("its text                :", repr(char_chunks[3].page_content))

Notice the chunk above: it knows its file and a character offset, but not its **section** — the "## Incidents"
heading went into a different chunk, and the text was cut mid-sentence. `MarkdownHeaderTextSplitter` splits on headings instead and records the heading path
in the metadata — document-aware chunking (`rg41`). The heading path is useful twice: as a filter / citation, and
prepended to the text so the embedding knows the topic.

In [ ]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "doc"), ("##", "section")])

chunks = []
for d in docs:
    for c in md_splitter.split_text(d.page_content):
        c.metadata["source"] = Path(d.metadata["source"]).name     # keep the file, add the headings
        chunks.append(c)

for c in chunks[:4]:
    print(f"{str(c.metadata):62s} | {c.page_content[:45]}...")
print(len(chunks), "section chunks")
assert all({"doc", "section", "source"} <= set(c.metadata) for c in chunks)

def label(d: Document) -> str:
    return f"{d.metadata['source']}#{d.metadata['section']}"

Real pipelines combine them: split by headings first, then recursively split any section that is still too long.
The histogram compares chunk lengths for the two strategies.

In [ ]:
two_stage = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30).split_documents(chunks)
print("heading split:", len(chunks), "chunks | + recursive(150):", len(two_stage), "chunks, all keep headings:",
      all("section" in c.metadata for c in two_stage))

plt.figure(figsize=(8, 3.3))
plt.hist([len(c.page_content) for c in char_chunks], bins=range(0, 260, 20), alpha=0.6, label="recursive, 150 chars")
plt.hist([len(c.page_content) for c in chunks], bins=range(0, 260, 20), alpha=0.6, label="by Markdown section")
plt.title("Chunk length by splitting strategy"); plt.xlabel("characters per chunk"); plt.ylabel("chunks")
plt.legend(); plt.tight_layout(); plt.show()

## 4. Embeddings

An `Embeddings` object has two methods: `embed_documents(list[str]) -> list[list[float]]` for indexing and
`embed_query(str) -> list[float]` for searching (some models embed queries differently from documents).

In [ ]:
emb = langchain_embeddings()
doc_vecs = np.array(emb.embed_documents([c.page_content for c in chunks]))
print("embedding model:", EMBED_MODEL, "| matrix:", doc_vecs.shape, "| norms ~1:", np.allclose(np.linalg.norm(doc_vecs, axis=1), 1))

queries = ["Can I deploy on a Friday?", "When can I take vacation?", "What does AUTH-4012 mean?",
           "Who can write to the production database?", "What is the capital of Peru?"]
q_vecs = np.array([emb.embed_query(q) for q in queries])
sims = q_vecs @ doc_vecs.T                       # unit vectors: dot product = cosine similarity

plt.figure(figsize=(10, 3.6))
plt.imshow(sims, cmap="viridis", aspect="auto")
plt.yticks(range(len(queries)), queries); plt.xticks(range(len(chunks)), [label(c) for c in chunks], rotation=60, ha="right", fontsize=8)
plt.colorbar(label="cosine similarity"); plt.title("Query × chunk similarity (bright = retrieved)")
plt.tight_layout(); plt.show()
print("best chunk per query:", [label(chunks[i]) for i in sims.argmax(axis=1)])

The last row matters: "capital of Peru" has *some* nearest chunk even though nothing is relevant. A vector search
always returns k results; relevance has to be decided by a threshold or by the LLM (`rg49`).

## 5. Vector stores: `InMemoryVectorStore` and FAISS

A vector store keeps (vector, document) pairs and answers "the k nearest vectors to this query".

| Store | Where the index lives | Good for |
|---|---|---|
| `InMemoryVectorStore` (`langchain_core`) | a Python dict, brute-force cosine | tests, demos, < ~10k chunks |
| `FAISS` (`langchain_community`) | a FAISS index in RAM, saved to disk | single-machine prototypes up to millions of vectors |
| pgvector, Qdrant, Pinecone, Elasticsearch... | a server | production: persistence, filtering, updates, access control |

All share the same interface: `from_documents`, `add_documents`, `similarity_search(_with_score)`, `delete`,
`as_retriever`.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.vectorstores import FAISS

mem_store = InMemoryVectorStore.from_documents(chunks, emb)
faiss_store = FAISS.from_documents(chunks, emb)

q = "Can I deploy on a Friday?"
for name, store in [("InMemory", mem_store), ("FAISS", faiss_store)]:
    print(name)
    for d, s in store.similarity_search_with_score(q, k=3):
        print(f"   score={s:.3f}  {label(d)}")
print("FAISS index:", type(faiss_store.index).__name__, "| vectors:", faiss_store.index.ntotal)

**Same ranking, different numbers — read the score before you threshold it.** `InMemoryVectorStore` returns cosine
*similarity* (higher is better). FAISS's default `IndexFlatL2` returns **squared L2 distance** (lower is better). For
unit-length vectors the two are tied exactly: ‖a−b‖² = 2 − 2·cos, so cos = 1 − d/2. The next cell proves it.

LangChain's *default* FAISS "relevance score" assumes a non-squared distance, so it produces negative values for
mildly related chunks, and a `score_threshold` then silently filters out good results. Fix: pass
`relevance_score_fn=lambda d: 1 - d / 2` (valid for normalised embeddings such as OpenAI's).

In [ ]:
d_faiss = faiss_store.similarity_search_with_score(q, k=1)[0][1]
cos_mem = mem_store.similarity_search_with_score(q, k=1)[0][1]
print(f"FAISS squared L2 = {d_faiss:.4f} -> 1 - d/2 = {1 - d_faiss / 2:.4f} | InMemory cosine = {cos_mem:.4f}")
assert abs((1 - d_faiss / 2) - cos_mem) < 1e-4

default_rel = [round(float(s), 3) for _, s in faiss_store.similarity_search_with_relevance_scores(q, k=4)]
faiss_store = FAISS.from_documents(chunks, emb, relevance_score_fn=lambda d: 1 - d / 2)
fixed_rel = [round(float(s), 3) for _, s in faiss_store.similarity_search_with_relevance_scores(q, k=4)]
print("default relevance scores:", default_rel)
print("fixed (cosine)          :", fixed_rel)
assert min(default_rel) < 0 <= min(fixed_rel)

### Saving and loading FAISS — and `allow_dangerous_deserialization`

`save_local` writes two files: `index.faiss` (the vectors) and `index.pkl` (the documents and id mapping, **pickled**).
Unpickling runs arbitrary code, so a malicious `index.pkl` can take over your machine. `load_local` therefore refuses
unless you pass `allow_dangerous_deserialization=True` — say yes only for files **you** created and stored somewhere
nobody else can write. Never for an index downloaded from the internet or uploaded by a user.

In [ ]:
idx_dir = Path(tempfile.mkdtemp()) / "handbook_index"
faiss_store.save_local(str(idx_dir))
print("files:", sorted(p.name for p in idx_dir.iterdir()))

try:
    FAISS.load_local(str(idx_dir), emb)
except ValueError as e:
    print("refused:", str(e)[:95], "...")

reloaded = FAISS.load_local(str(idx_dir), emb, allow_dangerous_deserialization=True,
                            relevance_score_fn=lambda d: 1 - d / 2)   # functions are not saved: pass it again
assert [label(d) for d in reloaded.similarity_search(q, k=3)] == [label(d) for d in faiss_store.similarity_search(q, k=3)]
print("reloaded:", reloaded.index.ntotal, "vectors, identical results")

One more trap (`rg14`): the query must be embedded with **the same model** as the documents. Vectors from another
model live in a different space (here even a different dimension — FAISS refuses; with the same dimension it would
silently return garbage). Changing the embedding model means re-embedding the corpus.

## 6. Retrievers: `as_retriever`

A retriever is a Runnable: string in, `list[Document]` out — the piece you pipe into a RAG chain.

| `search_type` | `search_kwargs` | Returns |
|---|---|---|
| `"similarity"` (default) | `k` | the k nearest chunks |
| `"mmr"` | `k`, `fetch_k`, `lambda_mult` (1 = pure relevance, 0 = max diversity) | k chunks that are relevant **and** different from each other |
| `"similarity_score_threshold"` | `score_threshold`, `k` | only chunks with relevance ≥ threshold (possibly none) |
| any of the above | `filter` | only chunks whose metadata matches |

In [ ]:
retriever = faiss_store.as_retriever(search_kwargs={"k": 2})
print(type(retriever).__name__, "->", [label(d) for d in retriever.invoke("Who can write to the production database?")])
print("batch:", [[label(d) for d in r] for r in retriever.batch(["Can I deploy on a Friday?", "When can I take vacation?"])])

thresholded = faiss_store.as_retriever(search_type="similarity_score_threshold",
                                       search_kwargs={"score_threshold": 0.2, "k": 3})
for query in ["Can I deploy on a Friday?", "When can I take vacation?", "What is the capital of Peru?"]:
    print(f"threshold 0.2 | {query:32s} -> {[label(d) for d in thresholded.invoke(query)]}")
assert thresholded.invoke("What is the capital of Peru?") == []      # (it also logs "No relevant docs...")

`InMemoryVectorStore` does not implement relevance scores in this version, so `similarity_score_threshold` raises
`NotImplementedError` on it — use FAISS (with the fixed `relevance_score_fn`) or a real vector DB for thresholds.

### MMR: stop returning the same fact three times

If the corpus has near-duplicates (the same policy copied into three pages), plain similarity fills all k slots with
them. MMR picks the most relevant chunk, then repeatedly the chunk that best balances "relevant to the query" against
"different from what I already picked".

In [ ]:
near_dupes = [
    Document("Friday deploys are blocked unless the change is tagged hotfix.", metadata={"id": "dup-1"}),
    Document("Deploys on a Friday are blocked unless the change is a hotfix.", metadata={"id": "dup-2"}),
    Document("Friday deploys are not allowed without a hotfix tag.", metadata={"id": "dup-3"}),
    Document("All production deploys require two approvals.", metadata={"id": "approvals"}),
    Document("Every deploy is announced in the releases channel.", metadata={"id": "announce"}),
    Document("On-call rotations are one week.", metadata={"id": "oncall"}),
]
dupe_store = InMemoryVectorStore.from_documents(near_dupes, emb)
question = "What are the rules for Friday deploys?"

plain = [d.metadata["id"] for d in dupe_store.as_retriever(search_kwargs={"k": 3}).invoke(question)]
mmr = [d.metadata["id"] for d in dupe_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5}).invoke(question)]
print("similarity:", plain)
print("mmr (0.5) :", mmr)
assert sum(i.startswith("dup") for i in plain) == 3 and sum(i.startswith("dup") for i in mmr) == 1

### Metadata filters

Filters restrict the search **before** ranking: per tenant, per permission, per product, per date (`rg10`). This is
also how you do permission-aware retrieval: never rely on the LLM to hide a chunk it was given. FAISS accepts a dict
(equality; newer versions also support operators), `InMemoryVectorStore` a function of the `Document`.

In [ ]:
question = "Who can write to the production database?"
print("no filter           :", [label(d) for d in faiss_store.similarity_search(question, k=3)])
print("FAISS source=infra  :", [label(d) for d in faiss_store.similarity_search(question, k=3, filter={"source": "infra.md"})])
print("InMemory, a function:", [label(d) for d in mem_store.similarity_search(
    question, k=3, filter=lambda d: d.metadata["source"] == "infra.md")])
only_infra = faiss_store.as_retriever(search_kwargs={"k": 3, "filter": {"source": "infra.md"}})
assert all(d.metadata["source"] == "infra.md" for d in only_infra.invoke(question))

## 7. BM25 and hybrid search with `EnsembleRetriever`

**BM25** is classic keyword search: it scores exact term matches, weighted by how rare the term is. It nails
identifiers, error codes and names, and knows nothing about synonyms. **Dense** retrieval matches meaning (here:
synonyms) but can blur rare tokens. Hybrid = run both, fuse the rankings.

`BM25Retriever`'s default tokenizer is `str.split` — case-sensitive and punctuation-glued ("Friday?" ≠ "Friday").
Always pass a `preprocess_func`.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever        # moved here in LangChain 1.x
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tokenize(text: str) -> list[str]:
    return [w for w in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if w not in ENGLISH_STOP_WORDS]

bm25 = BM25Retriever.from_documents(chunks, k=3, preprocess_func=tokenize)
dense = faiss_store.as_retriever(search_kwargs={"k": 3})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])    # weighted reciprocal rank fusion

tests = {"What does AUTH-4012 mean?": "security.md#Error codes",       # exact identifier
         "When can I take vacation?": "handbook.md#Leave",              # synonym: vacation ~ holiday
         "Can I deploy on a Friday?": "runbook.md#Deploys"}

def rank_of(gold, docs):
    labels = [label(d) for d in docs]
    return labels.index(gold) + 1 if gold in labels else None

results = {}
for query, gold in tests.items():
    results[query] = {name: rank_of(gold, r.invoke(query)) for name, r in [("bm25", bm25), ("dense", dense), ("hybrid", hybrid)]}
    print(f"{query:28s} rank of the right chunk -> {results[query]}")
print("BM25 scores for the vacation query:", bm25.vectorizer.get_scores(tokenize("When can I take vacation?")).max())

assert results["What does AUTH-4012 mean?"]["bm25"] == 1
assert results["When can I take vacation?"]["bm25"] is None                 # no shared words: BM25 is blind
assert all(r["hybrid"] is not None for r in results.values())               # hybrid finds every one

How `EnsembleRetriever` fuses: **reciprocal rank fusion**. Each retriever contributes `weight / (c + rank)` per
document (`c = 60` by default), and the sums are sorted. RRF uses ranks, not raw scores, so it does not care that BM25
scores are unbounded and cosine is in [-1, 1] — that is why it is the default (`rg45`). The trade-off: a document
found by only one lane lands lower (the vacation chunk comes 3rd, not 1st), which is why hybrid is usually followed
by a reranker ([reranking](../09_rag/05_reranking.ipynb)).

In [ ]:
names = list(tests)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 3.4))
for i, lane in enumerate(["bm25", "dense", "hybrid"]):
    ranks = [results[n][lane] or 0 for n in names]
    heights = [1 / r if r else 0 for r in ranks]
    ax.bar(x + (i - 1) * 0.27, heights, width=0.27, label=lane)
ax.set_xticks(x, [n[:26] for n in names]); ax.set_ylabel("1 / rank of the right chunk (0 = missed)")
ax.set_title("BM25 is blind to synonyms; the hybrid finds every question's chunk")
ax.set_ylim(0, 1.3); ax.legend(ncol=3, loc="upper center"); plt.tight_layout(); plt.show()

## 8. A custom `Embeddings` class

A favourite interview ask: "plug your own embedding model into LangChain". Subclass
`langchain_core.embeddings.Embeddings` and implement `embed_documents` and `embed_query` (the async versions default
to running these in a thread). Here: TF-IDF fitted on the corpus — no API, no download, fully deterministic. A
sentence-transformers wrapper would be the same ten lines with `model.encode(texts, normalize_embeddings=True)`.

Two details that bite: return plain `list[list[float]]`, and never return an all-zero vector (a query with no known
words) — cosine similarity divides by the norm and the store raises on NaN.

In [ ]:
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfEmbeddings(Embeddings):
    """TF-IDF vectors as LangChain embeddings. Fit once on the corpus; vocabulary is fixed after that."""

    def __init__(self, corpus: list[str]):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english").fit(corpus)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        m = self.vectorizer.transform(texts).toarray()
        m = np.hstack([m, np.full((len(texts), 1), 1e-3)])      # tiny constant dim: never an all-zero vector
        return (m / np.linalg.norm(m, axis=1, keepdims=True)).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

tfidf = TfidfEmbeddings([c.page_content for c in chunks])
tfidf_store = InMemoryVectorStore.from_documents(chunks, tfidf)
print("dimension:", len(tfidf.embed_query("hello")))
for query in ["Can I deploy on a Friday?", "What does AUTH-4012 mean?", "When can I take vacation?"]:
    d, s = tfidf_store.similarity_search_with_score(query, k=1)[0]
    print(f"{query:28s} -> {label(d):26s} cos={s:.3f}")
assert label(tfidf_store.similarity_search("Can I deploy on a Friday?", k=1)[0]) == "runbook.md#Deploys"

TF-IDF has no synonyms, so it misses the vacation question (near-zero score) exactly like BM25 — the interface is the
same, the model decides the quality. And the mixing trap from section 5, proved:

In [ ]:
try:
    faiss_store.similarity_search_by_vector(tfidf.embed_query("deploy on Friday"), k=1)
except AssertionError:
    print(f"FAISS refused a {len(tfidf.embed_query('x'))}-d TF-IDF query against a {faiss_store.index.d}-d index")

## Try it yourself

**1. Chunk-size sweep.** For `chunk_size` in 60, 100, 200, count the chunks `RecursiveCharacterTextSplitter`
(overlap 20) makes from the handbook and check that no chunk exceeds the size.

In [ ]:
# Solution — try it yourself first, then run this
for size in [60, 100, 200]:
    parts = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20).split_documents(docs)
    print(f"chunk_size={size:3d}: {len(parts):2d} chunks, longest {max(len(p.page_content) for p in parts)}")
    assert max(len(p.page_content) for p in parts) <= size

**2. Add and delete a document.** Add a new chunk about the office Wi-Fi to `mem_store` with an explicit id, show it
is retrieved, delete it by id, and show it is gone.

In [ ]:
# Solution — try it yourself first, then run this
wifi = Document("The office Wi-Fi password rotates every Monday and is posted in the lobby.",
                metadata={"source": "office.md", "section": "Wi-Fi", "doc": "Office"})
mem_store.add_documents([wifi], ids=["office-wifi"])
q_wifi = "What is the Wi-Fi password?"
print("after add   :", label(mem_store.similarity_search(q_wifi, k=1)[0]))
mem_store.delete(ids=["office-wifi"])
print("after delete:", label(mem_store.similarity_search(q_wifi, k=1)[0]))
assert label(mem_store.similarity_search(q_wifi, k=1)[0]) != "office.md#Wi-Fi"

**3. Re-weight the hybrid.** Give BM25 weight 0.2 and dense 0.8. Does the vacation chunk move up?

In [ ]:
# Solution — try it yourself first, then run this
dense_heavy = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.2, 0.8])
before = rank_of("handbook.md#Leave", hybrid.invoke("When can I take vacation?"))
after = rank_of("handbook.md#Leave", dense_heavy.invoke("When can I take vacation?"))
print(f"vacation chunk rank: weights 0.5/0.5 -> {before}, 0.2/0.8 -> {after}")
assert after <= before

## Say it in an interview

- **Pipeline:** loader → `Document`s (text + metadata) → splitter → embeddings → vector store → `as_retriever()`.
  The retriever is a Runnable: string in, `list[Document]` out.
- **Chunking:** split on document structure first (headings, `MarkdownHeaderTextSplitter`), then recursively by size
  with overlap; keep the heading path in metadata and in the text. Typical: a few hundred tokens, 10–20 % overlap —
  then measure.
- **Metadata** (source, section, tenant, permissions, updated_at) is what you filter, cite, delete and audit by.
  Filter before ranking; permissions are enforced by the retriever, never by the prompt.
- **Scores:** know what your store returns. FAISS default = squared L2 (lower better; cos = 1 − d/2 for unit vectors);
  LangChain's default FAISS relevance can go negative — pass `relevance_score_fn`. A vector search always returns k
  results; "nothing relevant" needs a threshold.
- **Hybrid:** BM25 for exact tokens (codes, names), dense for meaning; fuse with RRF (`EnsembleRetriever`, now in
  `langchain_classic.retrievers`), then rerank. MMR (`lambda_mult`) removes near-duplicates.
- **Custom embeddings:** subclass `Embeddings`, implement `embed_documents` / `embed_query`; same model for queries and
  documents, or re-embed everything.
- **Traps:** `allow_dangerous_deserialization=True` only for your own FAISS files (pickle); BM25Retriever's default
  tokenizer is `str.split`; `langchain_text_splitters` imports torch if sentence-transformers is installed.

## Next

- [05 · The RAG chain end to end](05_rag_chain_end_to_end.ipynb) — put this retriever behind a prompt and a model.
- Without a framework: [chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb),
  [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb), [vector search with FAISS/HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb),
  [hybrid search and RRF](../09_rag/04_hybrid_search_and_rrf.ipynb), [reranking](../09_rag/05_reranking.ipynb)
- Theory: [LangChain course](../../langchain/index.html) (chapters 6–7) · [interview bank](../../ai_interview_prep/index.html)